# 01 — First look at PAD000040

**Dataset:** *Inflammatory proteomic heterogeneity beyond glycemia status in severe obesity.*
Milito MM, ..., Giraudi PJ. Int J Mol Sci 2026;27(9):4152. doi:10.3390/ijms27094152.
PRIDE: [PAD000040](https://www.ebi.ac.uk/pride/archive/projects/PAD000040) — licence CC0 (public domain).

**Goals of this notebook**
1. Read the protein file (NPX) and understand its shape.
2. Read the sample list and **match the IDs** between the two files.
3. Look at the **quality warnings**.
4. Look at **values below the detection limit (LOD)**.

Every step that works here, and that we will need again, becomes a function in the package
(`signatomics.olink`). The list of what a good reader must handle is collected at the end.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

from signatomics.olink import read_npx, add_sample_info, qc_report, apply_qc  # our package
from signatomics.olink import plot_below_lod, plot_npx_strip, plot_group_by_plate, plot_sample_qc

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Find the signatomics folder, wherever the notebook is started from (VS Code may start it
# from a parent folder instead of analyses/). The data is in data/PAD000040/ (never in git).
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents, here / "signatomics"] if (p / "src" / "signatomics").is_dir())
DATA = ROOT / "data" / "PAD000040"
sorted(p.name for p in DATA.iterdir())

['2025.05.08_sample_manifest.xlsx',
 'PAD000040_community_annotated.sdrf.tsv',
 'dataset_all.npx.csv',
 'inflam_NASH_dt1.raw.csv',
 'inflam_PRA_dt2.raw.csv',
 'samplemanifest_bis.xlsx']

## 1. The protein file

Olink delivers data in **long format**: one row per *sample × protein*. With 92 proteins, each
sample measurement therefore takes 92 rows.

Things we found while exploring this file — all now handled by the package function
`read_npx`, which prints a **reading report** of everything it changed:
- separated by **commas** (other files use semicolons or tabs — detected automatically);
- `SampleID` must be read as **text** — IDs like `308_BIS` exist, and an ID such as `058` would
  otherwise lose its leading zero;
- some columns come from an Italian Excel: **`VERO` / `FALSO`** = TRUE / FALSE;
- `MissingFreq` mixes decimals (`0.89`) and percentages (`100%`).

In [2]:
npx = read_npx(DATA / "dataset_all.npx.csv")
npx.head(3).T

Reading report
  file: dataset_all.npx.csv
  file type: text, separated by comma
  layout: long (one row per sample and protein)
  MissingFreq: converted percentages
  AssayFlag: converted FALSO/VERO to True/False
  result: 16284 rows, 142 sample IDs, 92 proteins, 177 sample measurements, 2 plates


,0,1,2
SampleID,58,58,58
Index,1,1,1
OlinkID,OID00471,OID00472,OID05124
UniProt,P10145,P15692,P01732
Assay,IL8,VEGFA,CD8A
MissingFreq,0.0,0.0,0.0
Panel,Olink Inflammation,Olink Inflammation,Olink Inflammation
Panel_Version,Olink Target 96 Inflammation,Olink Target 96 Inflammation,Olink Target 96 Inflammation
PlateID,Inflammation F.I.F 21/03/2024,Inflammation F.I.F 21/03/2024,Inflammation F.I.F 21/03/2024
QC_Warning,Pass,Pass,Pass


In [3]:
runs = npx.drop_duplicates(["SampleID", "Index", "PlateID"])   # one row per sample measurement

print("rows                    :", len(npx))
print("proteins (assays)       :", npx["Assay"].nunique())
print("plates                  :", npx["PlateID"].nunique())
print("sample measurements     :", len(runs))
print("different sample IDs    :", npx["SampleID"].nunique())
runs["PlateID"].value_counts()

rows                    : 16284
proteins (assays)       : 92
plates                  : 2
sample measurements     : 177
different sample IDs    : 142


PlateID
Inflammation F.I.F 21/03/2024                  90
Inflammation PRAESIIDIUM Giraudi 04_03_2025    87
Name: count, dtype: int64

**Why 177 measurements but only 142 sample IDs?** Some samples were measured more than once:

- **34 samples are on both plates.** These are *bridging samples*: measured on both plates on
  purpose, so the two plates can be lined up with each other (bridging — a later notebook).
- **Sample 309 was measured twice on the same plate** — once with a quality warning, once without
  (a rerun).
- **`308_BIS`** is a repeat of sample 308 ("bis" = "again" in Italian) under a separate ID.

In [4]:
per_id = npx.groupby("SampleID").agg(measurements=("Index", "nunique"), plates=("PlateID", "nunique"))
print("on both plates          :", (per_id["plates"] > 1).sum())
print("twice on the same plate :", per_id[(per_id["measurements"] > 1) & (per_id["plates"] == 1)].index.tolist())
runs.loc[runs["SampleID"].isin(["308", "308_BIS", "309"]), ["SampleID", "Index", "PlateID", "QC_Warning"]]

on both plates          : 34
twice on the same plate : ['309']


,SampleID,Index,PlateID,QC_Warning
4232,309,78,Inflammation F.I.F 21/03/2024,Warning
8004,308,12,Inflammation F.I.F 21/03/2024,Pass
8096,309,24,Inflammation F.I.F 21/03/2024,Pass
16192,308_BIS,24,Inflammation F.I.F 21/03/2024,Pass


## 2. The sample list, and matching the IDs

The clinical information (diabetes status, sex, liver measures) is in a separate file, the
*sample manifest*. Two surprises:
- some column names end with a **space** (`"diabetes "`), so we strip them;
- the manifest has several ID columns. The one that matches the protein file is **`FIF ID`**
  (not `Subject_ID`, which is a well label like `Sample12`).

In [5]:
manifest = pd.read_excel(DATA / "2025.05.08_sample_manifest.xlsx")
manifest.columns = manifest.columns.str.strip()          # "diabetes " -> "diabetes"
manifest["FIF ID"] = manifest["FIF ID"].astype(str)
print(manifest.shape)
manifest.head()

(142, 11)


,WellID,FIF ID,PlateID,Subject_ID,sex,diabetes,fibrosi,fibrosis_diag,inflammation_diag,ballooning_diag,steatosis_diagnosis
0,A1,16,Plate 2,Sample1,F,No diabetes,1.75,1,1,1,1
1,C1,24,Plate 2,Sample25,F,No diabetes,2.00,1,1,1,1
2,D1,25,Plate 2,Sample37,F,No diabetes,1.75,1,1,1,1
3,E1,27,Plate 2,Sample49,F,No diabetes,2.00,1,1,1,0
4,H1,49,Plate 2,Sample85,F,No diabetes,1.75,1,0,1,1


In [6]:
# First attempt: compare the IDs exactly as they are written
protein_ids = set(npx["SampleID"])
manifest_ids = set(manifest["FIF ID"])

print("in both files            :", len(protein_ids & manifest_ids))
print("only in the protein file :", sorted(protein_ids - manifest_ids))
print("only in the sample list  :", sorted(manifest_ids - protein_ids))

in both files            : 141
only in the protein file : ['308_BIS']
only in the sample list  : ['308 BIS']


One ID does not match — but look closely: it is the **same sample**, written `308_BIS` (underscore)
in the protein file and `308 BIS` (space) in the sample list. So before matching we **tidy** the IDs:
remove spaces at the ends, use capitals, and treat spaces and underscores the same.

In [7]:
def tidy_sample_ids(ids):   # the rule add_sample_info uses: "308 BIS" == "308_bis" == "308_BIS"
    return ids.astype("string").str.strip().str.upper().str.replace(r"[\s_\-]+", "_", regex=True)

npx["sample"] = tidy_sample_ids(npx["SampleID"])
manifest["sample"] = tidy_sample_ids(manifest["FIF ID"])

protein_ids, manifest_ids = set(npx["sample"]), set(manifest["sample"])
print("in both files            :", len(protein_ids & manifest_ids))
print("only in the protein file :", sorted(protein_ids - manifest_ids))
print("only in the sample list  :", sorted(manifest_ids - protein_ids))

in both files            : 142
only in the protein file : []
only in the sample list  : []


**After tidying, all 142 match.** Without that step, one sample would have been silently dropped.

**Lessons for the package:** tidy IDs before matching, and always *report* what did not match —
silently losing samples is one of the most common hidden errors in omics analyses.

Note: `308_BIS` has its own row in the sample list, so it is treated as a separate sample here.
Whether it is a true repeat of 308 (same person) is something to confirm with the data owners.

### The diabetes groups — and which plate they were measured on

In [8]:
def plate_label(plates):
    if plates.nunique() > 1:
        return "both plates"
    return "plate 1 (2024)" if "F.I.F" in plates.iloc[0] else "plate 2 (2025)"

plate_of_sample = npx.groupby("sample")["PlateID"].agg(plate_label)
manifest["plate"] = manifest["sample"].map(plate_of_sample)

table = pd.crosstab(manifest["diabetes"], manifest["plate"], margins=True, margins_name="total")
table

plate,both plates,plate 1 (2024),plate 2 (2025),total
diabetes,,,,
No diabetes,12,35,22,69
Prediabetes,12,0,13,25
T2DM,10,20,18,48
total,34,55,53,142


In [9]:
groups = ["No diabetes", "Prediabetes", "T2DM"]
plates = ["plate 1 (2024)", "both plates", "plate 2 (2025)"]
colours = {"plate 1 (2024)": "#4C78A8", "both plates": "#9D9D9D", "plate 2 (2025)": "#F58518"}

fig = go.Figure()
for p in plates:
    counts = [table.loc[g, p] for g in groups]
    fig.add_bar(name=p, x=groups, y=counts, marker_color=colours[p],
                text=[c if c > 0 else "" for c in counts], textposition="inside")
fig.update_layout(
    barmode="stack", template="plotly_white", width=720, height=460,
    title=dict(text="Where each diabetes group was measured", x=0.02, y=0.97),
    yaxis_title="number of people",
    legend=dict(title="measured on:", orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=120),
)
fig.show()

**Important: no one with prediabetes was measured only on plate 1.** Prediabetes appears only on
plate 2 and among the bridging samples. So *group* and *plate* are partly mixed up: if plate 2
measures every protein a little higher, prediabetes would look "higher" too — without any biology.

This is called **confounding by batch**. Bridging the plates properly, and checking afterwards
that the plate difference is gone, is therefore not a detail but a requirement before any
biomarker search. It will be one of the package's built-in checks.

## 3. Quality warnings

Olink checks every sample measurement against internal controls. If a control is off, the
measurement gets `QC_Warning = "Warning"`. Usual practice: exclude warned measurements, or at
least check that results do not depend on them.

In [10]:
print(runs["QC_Warning"].value_counts())
pd.crosstab(runs["PlateID"], runs["QC_Warning"])

QC_Warning
Pass       160
Warning     17
Name: count, dtype: int64


QC_Warning,Pass,Warning
PlateID,,
Inflammation F.I.F 21/03/2024,85,5
Inflammation PRAESIIDIUM Giraudi 04_03_2025,75,12


In [11]:
# Missing NPX values: where are they?
empty = npx.groupby(["SampleID", "Index", "PlateID", "QC_Warning"])["NPX"].apply(lambda s: s.isna().mean())
print("missing NPX values in total :", npx["NPX"].isna().sum())
empty[empty > 0]

missing NPX values in total : 184


SampleID  Index  PlateID                                      QC_Warning
196       38     Inflammation PRAESIIDIUM Giraudi 04_03_2025  Warning       1.0
292       46     Inflammation PRAESIIDIUM Giraudi 04_03_2025  Warning       1.0
Name: NPX, dtype: float64

**17 of 177 measurements have a warning** (12 on plate 2, 5 on plate 1). All missing NPX values
come from **two measurements that are completely empty** (samples 196 and 292 on plate 2), both
with a warning. These two cannot be used at all; the other warned measurements have values but
deserve a sensitivity check.

## 4. Values below the detection limit (LOD)

Every protein has a **limit of detection (LOD)**: below it, the signal cannot be told apart from
background noise. Olink still reports a number, but it is unreliable. If a protein is below LOD
in most samples, it carries almost no information.

The file also has a `MissingFreq` column (share of samples below LOD per protein). In the file it
mixes decimals (`0.89`) and percentages (`100%`); `read_npx` already turned both into numbers
between 0 and 1.

In [12]:
print("MissingFreq is now a number from", npx["MissingFreq"].min(), "to", npx["MissingFreq"].max())

below = (
    npx.assign(below_lod=npx["NPX"] < npx["LOD"])
       .groupby("Assay")["below_lod"].mean()
       .sort_values(ascending=False)
)
print("proteins below LOD in more than 50% of measurements:", (below > 0.5).sum(), "of", len(below))
below.head(15).round(2)

MissingFreq is now a number from 0.0 to 1.0
proteins below LOD in more than 50% of measurements: 14 of 92


Assay
IL2           0.99
Beta-NGF      0.98
IL-20         0.93
IL33          0.91
LIF           0.86
IL-2RB        0.81
IL-1 alpha    0.80
IL4           0.77
IL5           0.67
IL13          0.66
IL-24         0.64
FGF-5         0.58
IL-22 RA1     0.54
TSLP          0.50
ARTN          0.46
Name: below_lod, dtype: float64

In [13]:
share = (below * 100).sort_values()
never = (share == 0).sum()
share = share[share > 0]                       # proteins never below LOD are left out of the chart
colour = ["#E45756" if v > 50 else "#4C78A8" for v in share]

fig = go.Figure(go.Bar(x=share.values, y=share.index, orientation="h", marker_color=colour))
fig.add_vline(x=50, line_dash="dash", line_color="#555555")
fig.add_annotation(x=50, y=0, yref="paper", text="50% line", showarrow=False, xanchor="left",
                   yanchor="bottom", xshift=6, bgcolor="white")
fig.update_layout(
    template="plotly_white", width=760, height=720,
    title=dict(text=f"Share of measurements below the detection limit<br>"
                    f"<sup>{len(share)} proteins shown; the other {never} are never below LOD</sup>", x=0.02),
    xaxis=dict(title="% of measurements below LOD", range=[0, 105]),
    yaxis=dict(tickfont=dict(size=11)),
    margin=dict(l=110, t=90),
)
fig.show()

**14 of 92 proteins are below the detection limit in more than half of the measurements** (red).
Four of them (IL2, Beta-NGF, IL-20, IL33) are below LOD almost always — exactly the four marked
`AssayFlag = VERO` (TRUE) in the file.

A common rule is to drop such proteins before the biomarker search. **We keep them all by
default**: a protein can be below the limit in one group and above it in another — then being
undetectable is itself a signal — and proteins without information do not come out of the
biomarker search anyway. The package can still apply a threshold, but only when asked.

In [14]:
# read_npx turned VERO/FALSO into True/False
print("proteins flagged in the file (AssayFlag = True):", npx.loc[npx["AssayFlag"].fillna(False), "Assay"].unique().tolist())

proteins flagged in the file (AssayFlag = True): ['IL2', 'Beta-NGF', 'IL-20', 'IL33']


## 5. What a good Olink reader must handle — learned from this one file

✅ = done in the package (`read_npx`, `add_sample_info`), with automatic tests.

| Issue found | What the package function should do | Status |
|---|---|---|
| IDs like `308_BIS`, possible leading zeros | always read sample IDs as text | ✅ |
| Italian Excel: `VERO` / `FALSO` | convert to True / False (also other languages) | ✅ |
| `MissingFreq` mixes `0.89` and `100%` | convert both to a number between 0 and 1 | ✅ |
| Column names with trailing spaces / name variants | strip them; standard names (`PlateLOD` → `LOD`) | ✅ |
| `No Data` written instead of an empty cell | treat as empty, and report it | ✅ |
| `308_BIS` vs `308 BIS` | **tidy IDs** (spaces, underscores, capitals) before matching | ✅ |
| Several ID columns in the sample list | match on a chosen column and **report** unmatched IDs | ✅ `add_sample_info` |
| Same sample measured twice (309, 308/308_BIS) | detect repeats and report them; let the user choose which to keep | ✅ `qc_report` / `apply_qc` |
| Bridging samples on two plates | recognise them (input for the bridging step) | ✅ recognised; bridging itself: next |
| Completely empty measurements | report and exclude | ✅ |
| Quality warnings | report per plate; exclude or keep as a visible choice | ✅ |
| Proteins mostly below LOD | report share per protein; threshold as a visible choice | ✅ |
| Group partly mixed up with plate | **warn** (here: prediabetes only on plate 2 / bridging samples) | ✅ `qc_report(group=...)` |

Also supported by `read_npx` (not needed for this file): older **wide** exports (NPX Manager
style, one row per sample with all proteins side by side), converted automatically.

## 6. The same steps with the package — in a few lines

Everything above, done with the package functions. Each one prints a report of what it found or
changed, so nothing happens silently.

1. `read_npx` — read the file (reading report)
2. `add_sample_info` — attach the sample list (matching report)
3. `qc_report` — look at quality, change nothing (quality report)
4. `apply_qc` — remove low-quality data, with the choices written out (list of removals)

In [15]:
data = read_npx(DATA / "dataset_all.npx.csv")
data = add_sample_info(data, DATA / "2025.05.08_sample_manifest.xlsx", on="FIF ID")

Reading report
  file: dataset_all.npx.csv
  file type: text, separated by comma
  layout: long (one row per sample and protein)
  MissingFreq: converted percentages
  AssayFlag: converted FALSO/VERO to True/False
  result: 16284 rows, 142 sample IDs, 92 proteins, 177 sample measurements, 2 plates
Matching report
  protein data: 142 sample IDs; sample list: 142 IDs (matched on 'SampleID' = 'FIF ID')
  matched: 142 (1 only after tidying the IDs, e.g. spaces vs underscores)
  columns that exist in both files got the ending _samplelist: PlateID


In [16]:
quality = qc_report(data, group="diabetes")

Quality report
  177 sample measurements of 142 sample IDs, 92 proteins, 2 plates
  quality warnings: 17 measurements
  WARNING completely empty measurements (no values at all): 196, 292
  measured more than once on the same plate (repeats): 309
  on more than one plate (bridging samples): 34
  proteins below the detection limit in more than 50% of measurements: 14 of 92
  no sample measurement looks odd (median and spread over all proteins within 3 SD)
  WARNING group mixed up with plate: no 'Prediabetes' measured only on Inflammation F.I.F 21/03/2024. Plate differences can look like group differences — bridge/normalise the plates and check before comparing groups.


In [17]:
quality.group_by_plate      # the table behind the group-vs-plate warning

PlateID,Inflammation F.I.F 21/03/2024,Inflammation PRAESIIDIUM Giraudi 04_03_2025,both/several plates
diabetes,,,
No diabetes,35,22,12
Prediabetes,0,13,12
T2DM,20,18,10


In [18]:
clean = apply_qc(
    data,
    drop_warnings=True,        # remove measurements with a quality warning
    drop_empty=True,           # remove measurements without values
    max_below_lod=None,        # keep ALL proteins (default); e.g. 0.5 would remove proteins
                               # below the detection limit in >50% of measurements
    repeats="prefer_pass_then_last",   # sample measured twice on one plate: keep the passed rerun
)

Quality steps applied
  removed 2 measurements (no values): 196, 292
  removed 15 measurements (quality warning): 102, 188, 197, 199, 20, 212, 214, 239, 246, 251, 288, 309, 35, 91
  kept: 160 measurements, 132 sample IDs, 92 proteins


**Note what the default choices cost:** removing all measurements with a quality warning removes
**10 people completely** (142 → 132 sample IDs). That may be the right choice — or too strict,
because a warning does not always mean the values are wrong. That is why every choice is a visible
argument: try `drop_warnings=False` and check whether your results change (a *sensitivity analysis*).

**Next step:** bridging the two plates (the 34 bridging samples), and checking afterwards that the
plate difference is gone — needed before comparing the diabetes groups.

## 7. Quality charts from the package

The same checks as pictures — each is one line. They use the standard table, so the detection
limits are already inside (also per plate). Sample IDs only appear in the hover text with
`show_ids=True`.

In [19]:
plot_below_lod(data, group="diabetes")    # share below the detection limit, per protein and group

**How to read this chart:** proteins are sorted by how much the groups differ (largest at the
top). No protein differs by 30 percentage points or more here. But look at the pattern:
**prediabetes is below the limit less often** for most of the top proteins (e.g. IL5: 51% vs
70–75%). That is not necessarily biology — prediabetes was measured on plate 2 and among the
bridging samples, and plate 2 reads higher. The plate mix-up shows up here too.

In [20]:
plot_npx_strip(data, group="diabetes", order="below_lod")   # hollow dots = below LOD

**How to read this chart:** every dot is one measurement. Hollow dots are below the detection
limit (the black dash) — they cannot be told apart from background noise. Proteins on the left
are mostly hollow: their values are mostly unreliable. Groups are side by side, so you can see
at a glance whether a protein differs between the diabetes groups (most do not, by eye).

In [21]:
plot_group_by_plate(data, "diabetes")     # the plate–group mix-up as a picture

In [22]:
plot_sample_qc(data)                      # Olink's standard sample check: median vs spread

**Sample check:** each dot is one measurement, its median over all proteins (x) against its
spread (y). A dot outside the dashed lines (3 standard deviations) would point to a technical
problem such as dilution. Here there are none — but notice the two plates form two shifted
clouds: plate 2 reads higher overall.

Is that biology or the plate? The 34 bridging samples answer it: they are the **same people**
on both plates, yet in this file they read on average about **0.55 NPX higher on plate 2**
(≈1.5 times). So this file — although it has an adjustment column — still contains a plate
difference. Together with the prediabetes mix-up, a plain group comparison here would be
distorted. Bridging (next notebook) has to remove it, and the check afterwards must show the
bridging samples agreeing.

In [23]:
both = data[data.groupby("SampleID")["PlateID"].transform("nunique") > 1]       # bridging samples
paired = both.pivot_table(index=["SampleID", "Assay"], columns="PlateID", values="NPX")
difference = paired.iloc[:, 1] - paired.iloc[:, 0]                              # plate 2 - plate 1
print(f"same samples, plate 2 minus plate 1: median {difference.median():.2f} NPX "
      f"(= {2 ** difference.median():.2f} times higher)")

same samples, plate 2 minus plate 1: median 0.55 NPX (= 1.46 times higher)
